# Sentiment Analysis

This notebook explores the **Sentiment Analysis** layer of Equity Lake:

- **`SentimentAnalyzer`** — VADER-based sentiment scoring for financial text
- **`analyze_sentiment_scores`** — Batch scoring function for DataFrames
- **`SentimentMethod`** / **`SentimentLabel`** — Enums for methods and labels
- FinBERT support planned for a future phase

**Prerequisites:** Run `equity ingest` and `equity news` to have data available.

In [1]:
import sys
sys.path.insert(0, "../../src")

import pandas as pd
import numpy as np

try:
    from equity_lake.sentiment import SentimentAnalyzer, analyze_sentiment_scores
    from equity_lake.sentiment.analyzer import SentimentMethod, SentimentLabel
    print("Sentiment modules loaded")
except Exception as e:
    print(f"Import error: {e}")

try:
    from equity_lake.storage.duckdb import EquityDataDB
    print("Storage module loaded")
except Exception as e:
    print(f"Storage import error: {e}")

try:
    import plotly.express as px
    import plotly.graph_objects as go
    print("Plotly loaded")
except Exception as e:
    print(f"Plotly import error: {e}")


Sentiment modules loaded
Storage module loaded
Plotly loaded


## 1. SentimentAnalyzer Demo

Analyze individual headlines with VADER. The analyzer returns:
- **compound**: Overall sentiment (-1.0 to 1.0)
- **label**: `positive` (>= 0.05), `negative` (<= -0.05), or `neutral`
- **neg / neu / pos**: Sub-scores from VADER

In [2]:
try:
    analyzer = SentimentAnalyzer(method="vader")

    headlines = [
        "Apple reports record quarterly earnings, beating analyst expectations",
        "Tesla shares plunge after disappointing delivery numbers",
        "Fed holds interest rates steady amid mixed economic signals",
        "NVIDIA announces breakthrough AI chip with 3x performance gains",
        "Market closes flat as investors await inflation data",
    ]

    for h in headlines:
        result = analyzer.analyze(h)
        print(f"\nHeadline: {h}")
        print(f"  Compound: {result['compound']:+.4f}  Label: {result['label']}")
        scores = result['scores']
        print(f"  neg={scores['neg']:.3f}  neu={scores['neu']:.3f}  pos={scores['pos']:.3f}")
except Exception as e:
    print(f"Error: {e}")


2026-06-09 09:26:20 [info     ] Initialized VADER sentiment analyzer



Headline: Apple reports record quarterly earnings, beating analyst expectations
  Compound: -0.4588  Label: negative
  neg=0.300  neu=0.700  pos=0.000

Headline: Tesla shares plunge after disappointing delivery numbers
  Compound: -0.2500  Label: negative
  neg=0.308  neu=0.481  pos=0.212

Headline: Fed holds interest rates steady amid mixed economic signals
  Compound: +0.4588  Label: positive
  neg=0.000  neu=0.727  pos=0.273

Headline: NVIDIA announces breakthrough AI chip with 3x performance gains
  Compound: +0.3400  Label: positive
  neg=0.000  neu=0.769  pos=0.231

Headline: Market closes flat as investors await inflation data
  Compound: +0.1027  Label: positive
  neg=0.000  neu=0.833  pos=0.167


## 2. Batch Analysis

Use `analyze_batch()` to score a list of financial headlines at once. Results are returned as a DataFrame.

In [3]:
try:
    financial_headlines = [
        "JPMorgan posts strong Q4 earnings, revenue up 12%",
        "Boeing cuts 737 MAX production targets amid supply chain issues",
        "Amazon Web Services announces major price cuts for cloud customers",
        "Oil prices surge on OPEC production cut extension",
        "Bitcoin rallies past $100K as institutional adoption grows",
        "Retail sales disappoint, consumer confidence drops to 6-month low",
        "Microsoft completes $69B Activision Blizzard acquisition",
        "Chinese tech stocks rally on regulatory easing signals",
        "European Central Bank signals pause in rate hike cycle",
        "Semiconductor stocks tumble on weak forward guidance from TSMC",
        "Google DeepMind achieves breakthrough in protein folding prediction",
        "US housing market shows signs of stabilization after 18-month downturn",
        "Uber reports first annual profit since going public",
        "Bond yields spike after hot jobs data dashes rate cut hopes",
        "Nike revenue misses estimates, warns of slowing global demand",
    ]

    batch_df = analyzer.analyze_batch(financial_headlines)
    print(f"Analyzed {len(batch_df)} headlines")
    print(f"Positive: {(batch_df['label'] == 'positive').sum()}")
    print(f"Negative: {(batch_df['label'] == 'negative').sum()}")
    print(f"Neutral:  {(batch_df['label'] == 'neutral').sum()}")
    batch_df
except Exception as e:
    print(f"Error: {e}")


Analyzed 15 headlines
Positive: 4
Negative: 6
Neutral:  5


## 3. Sentiment Score Distribution

Histogram of compound sentiment scores across all headlines.

In [4]:
try:
    if 'batch_df' in dir() and not batch_df.empty:
        fig = px.histogram(
            batch_df, x="compound", nbins=20,
            title="Distribution of VADER Compound Sentiment Scores",
            labels={"compound": "Compound Score (-1 to 1)"},
            color_discrete_sequence=["#636EFA"],
        )
        fig.add_vline(x=0.05, line_dash="dash", line_color="green",
                       annotation_text="Positive threshold")
        fig.add_vline(x=-0.05, line_dash="dash", line_color="red",
                       annotation_text="Negative threshold")
        fig.update_layout(showlegend=False)
        fig.show()
    else:
        print("No batch data available. Run previous cell first.")
except Exception as e:
    print(f"Error: {e}")


## 4. Sentiment Label Distribution

Pie chart showing the breakdown of positive, negative, and neutral labels.

In [5]:
try:
    if 'batch_df' in dir() and not batch_df.empty:
        label_counts = batch_df["label"].value_counts().reset_index()
        label_counts.columns = ["label", "count"]

        color_map = {"positive": "#2CA02C", "negative": "#D62728", "neutral": "#FF7F0E"}
        fig = px.pie(
            label_counts, values="count", names="label",
            title="Sentiment Label Distribution",
            color="label", color_discrete_map=color_map,
        )
        fig.update_traces(textposition="inside", textinfo="percent+label")
        fig.show()
    else:
        print("No batch data available.")
except Exception as e:
    print(f"Error: {e}")


## 5. Sentiment vs Price Correlation

Load news sentiment data from DuckDB, merge with price returns, and visualize the relationship between sentiment and next-day returns.

In [6]:
try:
    db = EquityDataDB()

    price_df = db.query("SELECT ticker, date, close FROM us_equity ORDER BY ticker, date")
    price_df["date"] = pd.to_datetime(price_df["date"])
    price_df["return_1d"] = price_df.groupby("ticker")["close"].pct_change()
    price_df["return_next"] = price_df.groupby("ticker")["return_1d"].shift(-1)

    news_df = db.query("SELECT * FROM us_news LIMIT 5000")
    db.close()

    if news_df.empty:
        print("No news data found. Run `equity news` to fetch news data first.")
    else:
        print(f"Loaded {len(news_df)} news articles")

        if "headline" in news_df.columns:
            scored_df = analyze_sentiment_scores(news_df, text_column="headline")
        elif "title" in news_df.columns:
            scored_df = analyze_sentiment_scores(news_df, text_column="title")
        else:
            text_col = [c for c in news_df.columns if any(k in c.lower() for k in ["head", "title", "text", "summary"])][0]
            scored_df = analyze_sentiment_scores(news_df, text_column=text_col)

        date_col = "date" if "date" in scored_df.columns else scored_df.columns[0]
        scored_df[date_col] = pd.to_datetime(scored_df[date_col])

        daily_sentiment = scored_df.groupby(date_col).agg(
            avg_sentiment=("sentiment_score", "mean"),
            article_count=("sentiment_score", "count"),
        ).reset_index()

        merged = daily_sentiment.merge(
            price_df[["date", "return_next"]].rename(columns={"date": date_col}),
            on=date_col, how="inner",
        ).dropna()

        if not merged.empty:
            fig = px.scatter(
                merged, x="avg_sentiment", y="return_next",
                title="News Sentiment vs Next-Day Market Returns",
                labels={"avg_sentiment": "Avg Daily Sentiment Score",
                        "return_next": "Next-Day Return (%)"},
                color="article_count", color_continuous_scale="Viridis",
                trendline="ols",
                hover_data={date_col: True, "article_count": True},
            )
            fig.update_layout(yaxis_tickformat=".2%")
            fig.show()

            corr = merged["avg_sentiment"].corr(merged["return_next"])
            print(f"\nCorrelation (sentiment vs next-day return): {corr:.4f}")
        else:
            print("No overlapping dates between sentiment and price data.")
except Exception as e:
    print(f"Error: {e}")
    print("This section requires both news and price data. Run `equity news` and `equity ingest`.")


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/krx_equity


/var/folders/zv/p_57kc9j1fb9xtj06cw1qb1c0000gn/T/ipykernel_21131/3100665026.py:6: FutureWarning:

The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

Query failed: Catalog Error: Table with name us_news does not exist!
Did you mean "duckdb_views"?

LINE 1: SELECT * FROM us_news LIMIT 5000
                      ^


No news data found. Run `equity news` to fetch news data first.


## 6. Sentiment Trend Over Time

Line chart of average daily sentiment score, revealing bullish/bearish trends.

In [7]:
try:
    if "scored_df" in dir() and not scored_df.empty:
        date_col = "date" if "date" in scored_df.columns else scored_df.columns[0]
        scored_df[date_col] = pd.to_datetime(scored_df[date_col])

        daily = scored_df.groupby(scored_df[date_col].dt.date).agg(
            avg_compound=("sentiment_score", "mean"),
            pos_count=("sentiment_label", lambda x: (x == "positive").sum()),
            neg_count=("sentiment_label", lambda x: (x == "negative").sum()),
            total=("sentiment_score", "count"),
        ).reset_index()
        daily.columns = [date_col, "avg_compound", "pos_count", "neg_count", "total"]

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=daily[date_col], y=daily["avg_compound"],
            mode="lines+markers", name="Avg Sentiment",
            line=dict(color="#636EFA", width=2),
        ))
        fig.add_hline(y=0, line_dash="dash", line_color="gray",
                      annotation_text="Neutral")
        fig.add_hrect(y0=-1, y1=-0.05, fillcolor="red", opacity=0.05)
        fig.add_hrect(y0=0.05, y1=1, fillcolor="green", opacity=0.05)
        fig.update_layout(
            title="Average Daily Sentiment Score Over Time",
            xaxis_title="Date", yaxis_title="Avg Compound Score",
        )
        fig.show()
    else:
        print("No scored data available. Run previous cells first.")
except Exception as e:
    print(f"Error: {e}")


No scored data available. Run previous cells first.


## 7. News Headlines Analysis

Load from the `us_news` view, score headlines, and show the most positive and negative headlines.

In [8]:
try:
    db = EquityDataDB()
    news_raw = db.query("SELECT * FROM us_news ORDER BY date DESC LIMIT 200")
    db.close()

    if news_raw.empty:
        print("No news data available. Run `equity news` first.")
    else:
        text_col = None
        for col in ["headline", "title", "text", "summary"]:
            if col in news_raw.columns:
                text_col = col
                break

        if text_col is None:
            print(f"Available columns: {list(news_raw.columns)}")
        else:
            scored = analyze_sentiment_scores(news_raw, text_column=text_col)

            print("=== Top 5 Most Positive Headlines ===")
            top_pos = scored.nlargest(5, "sentiment_score")[[text_col, "sentiment_score", "sentiment_label"]]
            for _, row in top_pos.iterrows():
                print(f"  [{row['sentiment_score']:+.4f}] {row[text_col][:100]}")

            print("\n=== Top 5 Most Negative Headlines ===")
            top_neg = scored.nsmallest(5, "sentiment_score")[[text_col, "sentiment_score", "sentiment_label"]]
            for _, row in top_neg.iterrows():
                print(f"  [{row['sentiment_score']:+.4f}] {row[text_col][:100]}")

            print(f"\nOverall sentiment stats:")
            print(scored["sentiment_score"].describe())
except Exception as e:
    print(f"Error: {e}")


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/krx_equity


Query failed: Catalog Error: Table with name us_news does not exist!
Did you mean "duckdb_views"?

LINE 1: SELECT * FROM us_news ORDER BY date DESC LIMIT 200
                      ^


No news data available. Run `equity news` first.


## 8. Next Steps

- **[05-feature-engineering-deep-dive](05-feature-engineering-deep-dive.ipynb)** — Build features from sentiment scores
- **[07-signal-scanning](07-signal-scanning.ipynb)** — Generate trading signals using sentiment
- **[10-validation-and-quality](10-validation-and-quality.ipynb)** — Validate sentiment data quality and consistency